# Notebook 04: Trải nghiệm Giao diện Trực quan Web Demo (Gradio)
**Dự án:** Vietnamese ASR → English Translation trên Kaggle T4 ×2

### Tính năng:
1. Tải tệp âm thanh WAV/MP3/FLAC hoặc ghi âm giọng nói trực tiếp qua Microphone.
2. Tự động chia đoạn bằng VAD (`EnergyVADSegmenter`) cho các file âm thanh dài.
3. Nhận dạng tiếng Việt bằng mô hình tốt nhất (P2: ViP-VL + Token Pooling + Temporal Prior).
4. Dịch thuật song ngữ tự động sang tiếng Anh bằng `vinai-translate-vi2en-v2`.
5. Tạo liên kết công khai `share=True` (Gradio Public Link) để người dùng bên ngoài có thể tương tác trực tiếp.

In [ ]:
# Chuẩn bị mã nguồn dự án và cài đặt dependencies cho Web UI
%cd /kaggle/working
# Cấu hình thư mục cache Hugging Face ghi vào /tmp để không bị lỗi Read-only /root/.cache
%env HF_HOME=/tmp/huggingface
%env HF_DATASETS_CACHE=/tmp/huggingface/datasets
%env TRANSFORMERS_CACHE=/tmp/huggingface/hub
%env TORCH_HOME=/tmp/torch
%env PYTHONPATH=/kaggle/working/vi_asr_project/src
import sys
from pathlib import Path

work_dir = Path("/kaggle/working/vi_asr_project")
if not (work_dir / "pyproject.toml").exists():
    !rm -rf /kaggle/working/vi_asr_project
    !git clone --recurse-submodules https://github.com/DongTienDinh/vi-asr-translate.git /kaggle/working/vi_asr_project

%cd /kaggle/working/vi_asr_project
!pip install -q -e .
!pip install -q gradio soundfile
src_dir = str(work_dir / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)


In [ ]:
# Khởi chạy Demo Web UI
from vi_asr.demo.app import build_demo
from vi_asr.inference.pipeline import BilingualCascadePipeline
from vi_asr.inference.transcribe import ASRInferencePipeline
from vi_asr.evaluation.cascade import VinAITranslator
from pathlib import Path

# Kiểm tra checkpoint đã huấn luyện
ckpt_path = Path("runs/p2_token_temporal/checkpoint_best.pt")
tok_path = Path("artifacts/tokenizer/bpe_1024.model")

if ckpt_path.exists() and tok_path.exists():
    print("Loading trained ASR model and VinAI translator...")
    asr_pipe = ASRInferencePipeline.from_checkpoint(ckpt_path, tok_path)
    translator = VinAITranslator()
    pipeline = BilingualCascadePipeline(asr_pipeline=asr_pipe, translator=translator)
    demo = build_demo(pipeline_fn=pipeline.process_file)
else:
    print("Checkpoint not found. Running demo in interactive verification mode...")
    demo = build_demo()

demo.launch(share=True, debug=True)